# 개인 의도 축 calibration

Quest 컨트롤러의 왕복 궤적으로 좌우 손의 위치 보정 행렬을 구합니다. 로봇 연결 없이 수집·계산·저장할 수 있습니다.

전진·왼쪽·상승·오른쪽·왼쪽＋상승·오른쪽＋상승을 각각 5회 수집합니다. 단일 방향의 1~4회는 학습, 5회째와 두 대각선 동작 전체는 검증에 사용합니다.

새 커널에서 위에서부터 실행하세요. 설치·연결·SIM/REAL 적용 방법은 [calibration 안내](../docs/CALIBRATION.md)를 참고하세요. PC와 Quest는 같은 네트워크에 연결하고, 다른 SIM/REAL/수집 창의 Quest 수신기는 먼저 STOP하세요.


In [ ]:
# Locate the deployed workspace from the notebook, project or repository directory.
from pathlib import Path
import runpy

_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
               for candidate in (base, base / "VR_teleoperation")]
_root = next((candidate for candidate in _candidates
              if (candidate / ".project_v2").is_file()), None)
if _root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_bootstrap = runpy.run_path(str(_root / "package" / "init_project.py"))
for _name in _bootstrap["__all__"]:
    globals()[_name] = _bootstrap[_name]

from ri_motion_v5_package.utility import get_ip_address
from package.intent_capture_ui import run_intent_capture

PROJECT_DIR = _root.resolve()
print("Calibration workspace:", PROJECT_DIR)


## 수집 — 6종 × 5회

Quest 높이·방향과 본인 위치를 고정하고 편한 팔꿈치 굽힌 자세에서 시작합니다. 두 손을 함께 같은 방향으로 옮기며 몸통·손목 방향은 유지합니다. 좌우 동작은 몸통 중심 회전이나 양팔 벌리기가 아닌 평행 이동입니다.

각 반복은 3초 준비 + 6초 기록입니다. 기록 시작과 끝에는 0.3초 이상 정지하고, 중간에 이동했다가 출발 위치로 돌아옵니다. 회전 허용량은 시작 컨트롤러 방향에서 30°입니다. 움직임을 크게 만들 필요는 없으며 편한 10~20cm 범위부터 시작합니다.

| 동작 | 구체적 수행 | 역할 |
|---|---|---|
| 전진 | 높이·양손 간격 유지, 앞으로 뻗고 복귀 | 학습 |
| 왼쪽 | 높이·앞뒤 거리 유지, 두 손을 함께 왼쪽(+Y)으로 옮기고 복귀 | 학습 |
| 상승 | 앞뒤·좌우 위치 유지, 두 손을 위로 올리고 복귀 | 학습 |
| 오른쪽 | 왼쪽과 대칭으로 두 손을 오른쪽(-Y)으로 옮기고 복귀 | 학습 |
| 왼쪽＋상승 | 왼쪽 위(+Y/+Z)로 대각선 이동 후 복귀 | 검증 전용 |
| 오른쪽＋상승 | 오른쪽 위(-Y/+Z)로 대각선 이동 후 복귀 | 검증 전용 |

학습 동작의 1~4회로 보정하고 5회째는 학습에서 제외해 검증합니다. 좌우는 방향 부호를 맞춰 같은 비중으로 좌우 축 학습에 반영합니다. 두 복합 동작은 모두 검증 전용입니다. 전진＋상승은 새 수집·학습·필수 검증에서 제외합니다.

실패한 회차는 완료 횟수에 포함되지 않습니다. 시작 버튼은 이미 완료한 반복을 유지하고 남은 회차를 수집합니다. ‘선택 동작만 다시 수집’은 그 동작의 완료 기록을 제외하고 새로 수집하는 버튼입니다.


In [ ]:
PC_HOST = get_ip_address()  # Quest에서 접근할 PC LAN IP. 자동 감지가 틀리면 직접 지정하세요.
QUEST_PORT = 8443
PREPARE_S = 3.0
CAPTURE_S = 6.0
MAX_WRIST_ROTATION_DEG = 30.0
RESUME_CAPTURE_PATH = None  # 새 사용자/새 Quest 세션은 처음부터 수집합니다.
# 같은 Quest 세션·origin·머리 기준을 유지한 경우만 저장 궤적 JSON 경로를 지정해 이어 수집합니다.

print("Quest Browser:", "https://%s:%d/" % (PC_HOST, QUEST_PORT))
print("수집 모드:", "새 calibration" if RESUME_CAPTURE_PATH is None else "저장 시연 이어 수집")


## 실행과 저장

아래 셀을 실행하면 수집 창과 Quest HTTPS 수신기가 시작됩니다. 표시된 주소를 Quest Browser에서 열고 VR 세션을 시작하세요. 인증서 안내가 나오면 본인 PC 주소인지 확인하고 연결을 진행합니다. 로봇 bringup/inbound/outbound는 필요 없습니다.

양손 모두 ‘추적 신호 정상’이고 ‘수집 시작 가능’이 표시되면 동작을 선택한 뒤 수집 버튼을 누르세요. 그립을 쥘 필요는 없습니다. 선택 동작의 5회를 완료한 뒤 다음 동작으로 넘어갑니다.

- ‘선택 동작만 다시 수집’: 해당 동작의 완료 기록을 제외하고 다시 수집합니다.
- ‘전체 기준/수집 초기화’: Quest 방향이나 본인 위치가 바뀌었을 때 새 기준으로 시작합니다.
- ‘검증 결과 미리보기’: 지금까지 모은 궤적의 자동 검증 결과를 표시합니다.
- **Calibration 이름**: 저장할 이름을 입력합니다. 예: `jaehoon` → `260923_jaehoon`. 날짜는 저장 시점의 PC 날짜이며, 영문(A–Z, a–z)·숫자·공백·`_`·`-`만 40자까지 입력할 수 있습니다. 같은 날짜·이름이 이미 있으면 다른 이름을 입력하세요.
- ‘저장·보정 계산 후 종료’: 입력한 이름으로 수집과 보정 결과를 저장합니다. 창을 닫아도 현재까지의 수집은 저장되며, 유효한 이름이 없으면 실행 시각 기반 이름으로 자동 저장합니다.

처음부터 수집하려면 `RESUME_CAPTURE_PATH=None`을 유지하세요. 이어 수집은 같은 Quest 세션·origin·머리 기준에서만 가능합니다. 기존 파일은 덮어쓰지 않습니다.


In [ ]:
capture_result = run_intent_capture(
    PROJECT_DIR, PC_HOST, port=QUEST_PORT,
    prepare_s=PREPARE_S, capture_s=CAPTURE_S,
    max_wrist_rotation_deg=MAX_WRIST_ROTATION_DEG,
    resume_path=RESUME_CAPTURE_PATH,
)


## 결과 확인과 적용

원본 궤적은 `calibration/diagnostics/`의 `YYMMDD_이름_trajectory.json`, 자동 검증을 통과한 프로필은 `calibration/profiles/`, 미완료·미통과 결과는 `calibration/diagnostics/` 아래에 `YYMMDD_이름.json`으로 저장됩니다. 원본 궤적에는 `_trajectory`가 붙습니다(예: `260923_jaehoon_trajectory.json`). 개인 궤적과 새 프로필은 Git에서 기본 제외됩니다.

`accepted`는 정해진 궤적 검사를 통과했는지를 뜻합니다. 실제 조작감·작업 성공률과는 별도로 확인하세요. 미통과 결과도 원본 궤적과 함께 남아 진단과 SIM 후보 시험에 사용할 수 있습니다.

아래 출력의 **프로필 경로**를 SIM/REAL 설정에 넣는 방법과 미통과 후보의 SIM 확인 방법은 [calibration 안내의 프로필 적용](../docs/CALIBRATION.md#결과-해석과-sim-적용)을 따르세요. 노트북의 기존 `v1` 경로는 자동으로 변경되지 않습니다.


In [ ]:
profile = capture_result["profile"]
print("수집 경로:", capture_result["capture_path"])
print("프로필 경로:", capture_result["profile_path"])
print("자동 검증:", profile["accepted"], "|", profile["reason"])
print("미통과 검사:", profile.get("failed_checks", []))
for hand, data in profile.get("hands", {}).items():
    print(hand, "조건수:", data["condition"])
    print("보정 행렬:", data["correction"])
    for check in data["checks"]:
        if check["role"] == "held_out":
            print(check["label"], check["repeat_index"], check["passed"], check["after"])
